# OGX: what Spark does with the asset-integrity lakehouse

The nightly CDE pipeline (`cde/dags/ogx_dag.py`) lands six sources, ingests them to bronze, extracts the
unstructured files, builds silver, the asset master and gold, and reconciles every layer. This notebook
opens that lakehouse in a **Spark session**, follows one day of drone video through it, re-runs the
pipeline's own extractor on the executors, and ends with Spark and the CAI corrosion model working
together.

| Step | What it shows | Spark capability |
|---|---|---|
| 1 | The lakehouse and the day's pipeline audit | Spark SQL over Iceberg, one catalog for every layer |
| 2 | One day of drone flights, bronze to silver | binary objects catalogued by hash, keyframes as rows |
| 3 | The extractor, re-run live | `binaryFile` + `mapPartitions`: the job's own code on the executors |
| 4 | The wrong-tag flight | GPS against the asset register, a distance join in Spark SQL |
| 5 | Iceberg on production tables, read-only | snapshots, history, files, time travel |
| 6 | Iceberg on a scratch table | create, schema evolution, MERGE, time travel, rollback |
| 7 | Spark + CAI: corrosion scoring on the executors | `mapInPandas` with the champion model and its guardrails |

**This notebook is a safe demo.** Production tables are only **read**. Everything it writes goes to a
scratch Iceberg catalog `ogx_demo` under `.../rsingh_ogx/notebook_scratch/`, outside the Hive metastore,
which the last cell deletes. It never calls a job's write path, never starts a CDE or CAI job and never
touches the endpoint. The code it runs is imported from the repository, so there is no "works in the
notebook, rewrite it for production" gap (docs/DRONE_VIDEO_SPARK.md explains the pipeline).

**Session:** New Session › JupyterLab, Python 3.11 › **Enable Spark: Spark 3.5.4 - CDE 1.25.2** › 2 vCPU /
8 GB. The session asks for two small executors. Avoid 02:00-03:00 IST, when the nightly drift and retrain
jobs read the same tables. The file reaches the CAI project with the next code sync (`ogx-00`), or run
`git pull` in the session's terminal (code only: data, models and outputs are git-ignored). On a laptop: `OGX_NB_LOCAL_WAREHOUSE=<warehouse of scripts/run_local.py>`.

The data is **synthetic** (an operator called OGX Energy, flights generated from a library of real
corrosion photographs). Say so up front.

> **Say:** "Same tables, same code as the nightly jobs. The only difference is that this session is not
> allowed to write to production."

In [ ]:
import glob
import json
import os
import shutil
import sys
import tempfile
import time
import zipfile
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
for p in (ROOT, ROOT / "cde" / "jobs", ROOT / "scripts"):
    sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import pandas as pd

CFG = json.loads((ROOT / "config" / "lakehouse.json").read_text())
P = CFG["db_prefix"]                                    # rsingh_ogx
DEMO_DATE = os.environ.get("OGX_NB_DATE", "2026-10-06")  # the day of the wrong-tag flight
LOCAL = os.environ.get("OGX_NB_LOCAL_WAREHOUSE")        # laptop: a scripts/run_local.py warehouse
if LOCAL:
    SCRATCH = (Path(LOCAL).resolve().parent / "notebook_scratch").as_uri()
else:
    SCRATCH = CFG["landing"].rsplit("/", 1)[0] + "/notebook_scratch"
assert SCRATCH.rstrip("/").endswith("notebook_scratch"), SCRATCH
BUCKET = "/".join(CFG["landing"].split("/")[:3])
DEMO = "ogx_demo"                                        # the scratch catalog
t = lambda layer, name: f"{P}_{layer}.{name}"
pd.set_option("display.max_colwidth", 90)
print(f"repo: {ROOT.name} | date {DEMO_DATE} | {'laptop warehouse ' + LOCAL if LOCAL else 'CAI + data lake'}")
print(f"scratch (the only place this notebook writes): {SCRATCH}")

## 0. The Spark session

**On CAI** the Spark add-on ships the Iceberg runtime in `/opt/spark/optional-lib`; `SparkSessionCatalog`
puts Iceberg in front of the Hive metastore, so the tables resolve by the names the CDE jobs use.
`spark.kerberos.access.hadoopFileSystems` gets the session an IDBroker token for the bucket.
**On a laptop** the session is `scripts/run_local.py`'s, the one the local pipeline ran with.

Either way a second catalog, `ogx_demo`, is an Iceberg *Hadoop* catalog on the scratch folder: tables
there are plain files, invisible to Impala and Hue, and gone when the folder is deleted.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

if LOCAL:
    import run_local
    spark = run_local.local_spark(Path(LOCAL))
else:
    jars = ",".join(sorted(glob.glob("/opt/spark/optional-lib/iceberg-*.jar")))
    builder = (SparkSession.builder.appName("ogx-spark-demo-notebook")
               .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
               .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog")
               .config("spark.sql.catalog.spark_catalog.type", "hive")
               .config("spark.hadoop.iceberg.engine.hive.enabled", "true")
               .config("spark.kerberos.access.hadoopFileSystems", BUCKET)
               .config("spark.executor.instances", "2").config("spark.executor.cores", "1")
               .config("spark.executor.memory", "3g"))
    if jars:
        builder = builder.config("spark.jars", jars)
    spark = builder.getOrCreate()

spark.conf.set(f"spark.sql.catalog.{DEMO}", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set(f"spark.sql.catalog.{DEMO}.type", "hadoop")
spark.conf.set(f"spark.sql.catalog.{DEMO}.warehouse", SCRATCH + "/warehouse")

import ogx_common as C   # the CDE jobs' helpers: session settings, file system, table names
C.configure(spark)
print("Spark", spark.version, "| master", spark.sparkContext.master,
      "| default parallelism", spark.sparkContext.defaultParallelism)

**Ship the repository's code to the executors.** One zip of `extract/`, `features/`, `guardrails/` and the
two job modules, added with `addPyFile`: the executors then import exactly what the CDE jobs and the CAI
model import. The probe asks each executor which libraries it has; section 7 needs numpy, Pillow and
scikit-learn there.

In [ ]:
CODE_ZIP = Path(tempfile.gettempdir()) / "ogx_notebook_code.zip"
with zipfile.ZipFile(CODE_ZIP, "w") as zf:
    for pkg in ("extract", "features", "guardrails"):
        for f in (ROOT / pkg).rglob("*.py"):
            zf.write(f, f.relative_to(ROOT).as_posix())
    for f in ("ogx_common.py", "extract_unstructured.py"):
        zf.write(ROOT / "cde" / "jobs" / f, f)
spark.sparkContext.addPyFile(str(CODE_ZIP))
sys.path[:] = [p for p in sys.path if not p.endswith(CODE_ZIP.name)]   # the driver imports from the checkout


def probe(_):
    import importlib
    import platform
    import socket
    out = {"host": socket.gethostname(), "python": platform.python_version()}
    for m in ("numpy", "PIL", "sklearn", "joblib", "extract.video_keyframes", "features.feature_logic"):
        try:
            out[m] = getattr(importlib.import_module(m), "__version__", "ok")
        except Exception as e:  # noqa: BLE001
            out[m] = f"MISSING ({type(e).__name__})"
    yield out


probe_df = pd.DataFrame(spark.sparkContext.parallelize(range(2), 2).mapPartitions(probe).collect())
EXECUTORS_CAN_SCORE = not probe_df.astype(str).apply(lambda c: c.str.startswith("MISSING")).any().any()
display(probe_df)
print("executors can run the corrosion model:", EXECUTORS_CAN_SCORE)

## 1. The lakehouse at a glance

**What:** every database of the project, its tables, and what the pipeline recorded for the demo date in
`ref.load_audit`.

**Why:** one Spark session sees every layer (bronze, silver, asset master, gold, semantic, ref) through
one catalog, the same tables Impala, the dashboards and both CAI applications read.

**How:** `SHOW TABLES` per database, then plain Spark SQL on the audit table.

In [ ]:
layers = ["bronze", "silver", "asset", "gold", "semantic", "ref"]
inventory = []
for layer in layers:
    try:
        names = [r["tableName"] for r in spark.sql(f"SHOW TABLES IN {P}_{layer}").collect()]
    except Exception:  # noqa: BLE001
        names = []
    inventory.append({"database": f"{P}_{layer}", "tables": len(names), "examples": ", ".join(sorted(names)[:6])})
display(pd.DataFrame(inventory))

audit = spark.sql(f'''
    SELECT stage, entity, status, rows_in, rows_out, rows_rejected,
           CAST(ended_at AS STRING) AS ended_at, substr(message, 1, 80) AS message
    FROM {t("ref", "load_audit")}
    WHERE business_date = DATE '{DEMO_DATE}' AND status IN ('COMPLETED', 'FAILED')
    ORDER BY ended_at''')
audit.toPandas()

## 2. One day of drone flights: bronze to silver

**What:** the drone files of the date as bronze catalogued them, and the keyframes extraction made of them.

**Why:** a video is not loaded into a table; it is **validated, fingerprinted and stored once by its
sha256** in the raw store, and the table holds the facts and the path. The keyframes are then ordinary
rows: frame time, hash, GPS, a sharpness proxy. That is what makes video joinable with work orders and
inspection reports.

**How:** `bronze.doc_object` (one row per file) and `silver.video_keyframe` (one row per keyframe).

In [ ]:
flights = spark.sql(f'''
    SELECT file_name, `format`, ingest_status, reject_reason, size_bytes, asset_hint, hint_source,
           CAST(captured_at AS STRING) AS captured_at, substr(doc_id, 1, 12) AS sha256_12
    FROM {t("bronze", "doc_object")}
    WHERE source = 'drone' AND _business_date = DATE '{DEMO_DATE}'
    ORDER BY file_name, `format`''')
display(flights.toPandas())

kf = spark.table(t("silver", "video_keyframe")).where(F.col("business_date") == DEMO_DATE)
per_flight = (kf.groupBy("video_id", "asset_hint", "file_name")
              .agg(F.count("*").alias("keyframes"), F.max("t_seconds").alias("last_t_s"),
                   F.round(F.min("bytes_per_kpx"), 1).alias("min_bytes_per_kpx"),
                   F.round(F.avg("lat"), 5).alias("lat"), F.round(F.avg("lon"), 5).alias("lon"))
              .orderBy("file_name"))
per_flight.toPandas()

**The sharpness proxy, per keyframe.** Without an image decoder in the CDE jobs, JPEG bytes per 1,000 pixels
stands in for sharpness: a blurred frame compresses smaller. One flight a day carries two unfit frames;
look for the dips. Section 7 measures the same frames properly.

In [ ]:
pdf = kf.select("video_id", "t_seconds", "bytes_per_kpx").orderBy("video_id", "t_seconds").toPandas()
fig, ax = plt.subplots(figsize=(10, 3.5))
for vid, g in pdf.groupby("video_id"):
    ax.plot(g["t_seconds"], g["bytes_per_kpx"], marker="o", label=vid)
ax.set_xlabel("seconds into the flight")
ax.set_ylabel("JPEG bytes per kilopixel")
ax.set_title(f"Keyframes of {DEMO_DATE}: sharpness proxy")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. The extractor, re-run live on the executors

**What:** the day's videos read again from the raw store and parsed by
`extract_unstructured.extract_partition`, the function the CDE job runs, then compared with
`silver.video_keyframe`.

**Why:** the bytes never pass through the driver on the way in. `binaryFile` reads them where they lie
(S3 here), and each executor parses its share in a `mapPartitions`. The same code scales from four
flights to thousands by adding executors, not by rewriting it.

**How:** exactly the job's steps (sidecars on the driver, videos on the executors), but the result is a
DataFrame in memory; **nothing is written**. The keyframe JPEGs stay on the executors.

In [ ]:
import extract_unstructured as X

objs = spark.table(t("bronze", "doc_object")).where(
    (F.col("_business_date") == DEMO_DATE) & (F.col("ingest_status") == "VALID") & (F.col("source") == "drone"))
meta_rows = objs.select("doc_id", "source", "file_name", "format", "raw_path").collect()
fs = C.filesystem(spark, meta_rows[0]["raw_path"] if meta_rows else SCRATCH)
sidecars = {m["file_name"][:-5]: json.loads(fs.read_bytes(m["raw_path"]))
            for m in meta_rows if m["format"] == "json"}
metas = [(m["raw_path"], json.dumps({"doc_id": m["doc_id"], "source": m["source"], "file_name": m["file_name"],
                                     "format": m["format"], "sidecar": sidecars.get(m["file_name"][:-4])}))
         for m in meta_rows if m["format"] == "avi"]
print(f"{len(metas)} videos, {len(sidecars)} sidecars")

t0 = time.time()
meta_df = spark.createDataFrame(metas, "path string, meta string")
content = spark.read.format("binaryFile").load([p for p, _ in metas]).select("path", "content")
joined = X._join_paths(content, meta_df).repartition(max(2, len(metas) // 2))


def keyframes_only(rows):
    for table, payload in X.extract_partition(rows):
        if table == "video_keyframe":
            r = json.loads(payload)
            r.pop("_jpeg", None)          # the image stays on the executor
            yield (json.dumps(r),)


live = (spark.createDataFrame(joined.rdd.mapPartitions(keyframes_only), "j string")
        .select(F.from_json("j", X.SCHEMAS["video_keyframe"]).alias("r")).select("r.*")
        .withColumn("spark_partition", F.spark_partition_id())).cache()
n_live = live.count()
print(f"re-extracted {n_live} keyframes from {len(metas)} videos in {time.time() - t0:.1f} s "
      f"across {joined.rdd.getNumPartitions()} partitions")

key = ["doc_id", "frame_index", "frame_sha256"]
silver_keys = kf.select(*key)
print("in the live run, not in silver:", live.select(*key).subtract(silver_keys).count(),
      "| in silver, not in the live run:", silver_keys.subtract(live.select(*key)).count())
live.groupBy("spark_partition").agg(F.countDistinct("doc_id").alias("videos"),
                                    F.count("*").alias("keyframes")).orderBy("spark_partition").toPandas()

> **Say:** "Zero differences both ways: the notebook and last night's job produced the same keyframes from
> the same bytes, because it is the same function."

## 4. The wrong-tag flight: GPS against the asset register

**What:** on 2026-10-06 the flight over PL-03-SEG-014 is named and hinted as PL-03-SEG-041. The asset
master trusted the GPS and put the conflict in its review queue. Here Spark recomputes that decision from
the raw facts: each flight's mean position against every pipeline segment.

**Why:** a geospatial join is just a cross join, a haversine expression and a window, so it runs
distributed with no extra library.

**How:** `asset.review_queue` and `asset.match_pair` for what the pipeline decided, then the recomputation
on `silver.video_keyframe` and `asset.golden_asset`.

In [ ]:
display(spark.sql(f'''
    SELECT src_system, src_name, reason, suggested_asset_id, status, detail
    FROM {t("asset", "review_queue")}
    WHERE as_of_date = DATE '{DEMO_DATE}' AND reason = 'NAME_GPS_CONFLICT' ''').toPandas())

segments = (spark.table(t("asset", "golden_asset"))
            .where((F.col("as_of_date") == DEMO_DATE) & (F.col("asset_class") == "PIPE_SEGMENT")
                   & F.col("latitude").isNotNull())
            .select("asset_id", F.col("tag").alias("segment"), "latitude", "longitude"))
tracks = (kf.groupBy("doc_id", "video_id", "asset_hint")
          .agg(F.avg("lat").alias("lat"), F.avg("lon").alias("lon")))

R = 6371000.0
dist = F.lit(2 * R) * F.asin(F.sqrt(
    F.pow(F.sin(F.radians(F.col("latitude") - F.col("lat")) / 2), 2)
    + F.cos(F.radians("lat")) * F.cos(F.radians("latitude"))
    * F.pow(F.sin(F.radians(F.col("longitude") - F.col("lon")) / 2), 2)))
ranked = (tracks.crossJoin(F.broadcast(segments)).withColumn("distance_m", F.round(dist, 0))
          .withColumn("rank", F.row_number().over(Window.partitionBy("doc_id").orderBy("distance_m"))))
nearest = ranked.where("rank = 1").select("video_id", F.col("asset_hint").alias("file_name_says"),
                                          F.col("segment").alias("gps_says"), "distance_m")
nearest = nearest.withColumn(
    "verdict", F.when(F.col("distance_m") > 500, "no segment within 500 m: resolved on its tag")
                .when(F.col("file_name_says") == F.col("gps_says"), "agree")
                .otherwise("CONFLICT: GPS wins"))
nearest.orderBy("video_id").toPandas()

Flights over flare stacks and tanks have no pipeline segment nearby (hundreds of metres or more); the
asset master only applies the GPS rule within 500 m (`GPS_BAND_M`), so those resolve on their tag.

## 5. Iceberg on the production tables (read-only)

**What:** the snapshot history of `silver.video_keyframe`, its data files, and a time-travel read of the
first snapshot.

**Why:** every nightly run is an atomic Iceberg commit replacing one business date's partition; nothing
is overwritten in place, so any earlier state can be read back. That is the audit answer to "what did the
model see on that night?".

**How:** Iceberg metadata tables (`.snapshots`, `.files`) and `VERSION AS OF`. Reads only.

In [ ]:
KF = t("silver", "video_keyframe")
snaps = spark.sql(f"SELECT committed_at, snapshot_id, operation, summary['added-records'] AS added, "
                  f"summary['total-records'] AS total FROM {KF}.snapshots ORDER BY committed_at")
display(snaps.toPandas())

display(spark.sql(f"SELECT partition, count(*) AS files, sum(record_count) AS records, "
                  f"round(sum(file_size_in_bytes) / 1024, 1) AS kb FROM {KF}.files GROUP BY partition "
                  f"ORDER BY 1").toPandas())

first = snaps.orderBy("committed_at").first()
if first is not None:
    then = spark.sql(f"SELECT count(*) AS n FROM {KF} VERSION AS OF {first['snapshot_id']}").first()["n"]
    print(f"keyframes at the first snapshot ({first['snapshot_id']}): {then} | now: {spark.table(KF).count()}")

## 6. Iceberg on a scratch table: the write side

**What:** a flight summary built from the tables above, written to `ogx_demo.drone.flight_summary`, then
evolved: a column added, rows merged, the history read back and the table rolled back.

**Why:** these are the operations the pipeline relies on (atomic commits, `MERGE`, schema evolution,
rollback), shown on a table nobody else reads.

**How:** the scratch catalog only. The production tables are the source, never the target.

In [ ]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {DEMO}.drone")
FS = f"{DEMO}.drone.flight_summary"
spark.sql(f"DROP TABLE IF EXISTS {FS}")
xref = (spark.table(t("asset", "match_pair"))          # per video: the asset it resolved to, and how
        .where((F.col("as_of_date") == DEMO_DATE) & (F.col("src_system") == "drone") & (F.col("decision") == "MATCHED"))
        .select(F.col("src_key").alias("doc_id"), "asset_id", F.col("rule").alias("resolved_by"))
        .dropDuplicates(["doc_id"]))
summary = (kf.groupBy("doc_id", "video_id", "asset_hint", "business_date")
           .agg(F.count("*").alias("keyframes"), F.round(F.min("bytes_per_kpx"), 1).alias("min_bytes_per_kpx"))
           .join(xref, "doc_id", "left"))
summary.writeTo(FS).using("iceberg").partitionedBy("business_date").create()
v1 = spark.sql(f"SELECT snapshot_id FROM {FS}.snapshots ORDER BY committed_at DESC LIMIT 1").first()[0]
print("created, snapshot", v1)

spark.sql(f"ALTER TABLE {FS} ADD COLUMN review_note string")          # schema evolution, no rewrite
spark.sql(f'''
    MERGE INTO {FS} t
    USING (SELECT doc_id, 'GPS and file name disagree: resolved by GPS' AS note
           FROM {FS} WHERE asset_hint IS NOT NULL) s
    ON t.doc_id = s.doc_id
    WHEN MATCHED AND t.asset_id IS NOT NULL THEN UPDATE SET review_note = s.note''')
display(spark.table(FS).orderBy("video_id").toPandas())
spark.sql(f"SELECT committed_at, snapshot_id, operation FROM {FS}.snapshots ORDER BY committed_at").toPandas()

The `MERGE` above notes **every** flight, which is wrong: only the wrong-tag flight needed a note. Read the
table as it was before, then roll it back to that snapshot. In production this is how a bad nightly
commit is undone in seconds, without restoring files.

In [ ]:
then = spark.sql(f"SELECT * FROM {FS} VERSION AS OF {v1}")   # read with the schema of that snapshot
print("columns at the first snapshot:", then.columns)
print("columns now:                  ", spark.table(FS).columns)
print("rows with a note now:", spark.sql(f"SELECT count(review_note) FROM {FS}").first()[0], "of", spark.table(FS).count())
spark.sql(f"CALL {DEMO}.system.rollback_to_snapshot('drone.flight_summary', {v1})")
print("after rollback:", spark.sql(f"SELECT count(*) AS rows, count(review_note) AS noted FROM {FS}").first().asDict())
spark.sql(f"SELECT made_current_at, snapshot_id, is_current_ancestor FROM {FS}.history ORDER BY made_current_at").toPandas()

## 7. Spark + CAI: corrosion scoring on the executors

**What:** every keyframe of the day scored by the corrosion champion behind its guardrails, on the
executors, with `mapInPandas`. The frames are read from `raw/frames/<sha256>.jpg`, where the extract job
stored them.

**Why:** this is the limit of Spark alone (docs/DRONE_VIDEO_SPARK.md): the CDE jobs have no image decoder
and no model. A CAI Spark session has both, so Spark's scale and CAI's model meet in one step. It is a
prototype of the planned batch job `ogx-06` (`gold.fact_keyframe_score`); here it writes only to scratch.

**How:** the driver loads `models/champion` (installed by `ogx-04`), checks the feature hash, and broadcasts
the model, its metadata and the guardrail config. Each executor runs the same functions as the CAI
endpoint (`serve/predict.py`): `frame_checks`, the 49 features, the out-of-distribution check, the model,
`output_policy.apply` with the asset's criticality. If the probe found the libraries missing on the
executors, the same function runs on the driver instead, and says so.

In [ ]:
import yaml
from features.feature_logic import feature_hash

CHAMP = ROOT / os.environ.get("OGX_CHAMPION_DIR", "models/champion")
HAVE_CHAMPION = (CHAMP / "model.joblib").exists()
if not HAVE_CHAMPION:
    print(f"no champion in {CHAMP}: run the CAI chain ogx-01 to ogx-04 first; section 7 is skipped")
else:
    META = json.loads((CHAMP / "model_meta.json").read_text())
    assert META["feature_hash"] == feature_hash(), "champion was trained on different feature logic"
    GCFG = yaml.safe_load((ROOT / "config" / "guardrails.yaml").read_text())
    BC = spark.sparkContext.broadcast(((CHAMP / "model.joblib").read_bytes(), META, GCFG))
    print(f"champion: {META['estimator']}, feature version {META['feature_version']}, "
          f"threshold {META['threshold']:.3f}, test AUROC {META['metrics']['test']['auroc']:.3f}")

SCORE_SCHEMA = ("frame_sha256 string, criticality string, p_severe double, severity string, band string, "
                "band_reason string, failed_guardrails string")


def score_frames(batches):
    import io

    import joblib
    import pandas as pd

    from features.feature_logic import features, load_image
    from guardrails import input_checks, output_policy

    model_bytes, meta, cfg = BC.value
    model = joblib.load(io.BytesIO(model_bytes))
    for pdf in batches:
        rows = []
        for r in pdf.itertuples(index=False):
            img = load_image(r.content)
            checks = input_checks.frame_checks(img, cfg)
            x = features(img)
            checks.append(input_checks.ood_check(x, meta.get("ood")))
            p_sev, severity = None, None
            if all(c["passed"] for c in checks):
                proba = dict(zip(map(str, model.classes_), model.predict_proba(x.reshape(1, -1))[0]))
                p_sev, severity = float(proba.get("severe", 0.0)), max(proba, key=proba.get)
            out = output_policy.apply(p_sev, float(meta["threshold"]), float(meta["p1_probability"]), checks,
                                      r.criticality, cfg)
            rows.append({"frame_sha256": r.frame_sha256, "criticality": r.criticality, "p_severe": p_sev,
                         "severity": severity, "band": out["band"], "band_reason": out["band_reason"],
                         "failed_guardrails": ",".join(c["guardrail"] for c in checks if not c["passed"])})
        yield pd.DataFrame(rows, columns=[c.split(" ")[0] for c in SCORE_SCHEMA.split(", ")])

In [ ]:
if HAVE_CHAMPION:
    crit = (spark.table(t("gold", "dim_asset")).where("is_current").select("asset_id", "criticality"))
    frames = (kf.join(xref, "doc_id", "left").join(crit, "asset_id", "left")
              .select("frame_sha256", "frame_path", "video_id", "asset_id", "criticality").dropDuplicates(["frame_sha256"]))
    paths = [r["frame_path"] for r in frames.select("frame_path").collect()]
    blobs = spark.read.format("binaryFile").load(paths).select(F.col("path").alias("frame_path_q"), "content")
    base = lambda c: F.regexp_extract(c, r"([^/]+)$", 1)   # <sha256>.jpg: unique, scheme- and mount-proof
    to_score = (frames.join(blobs, base(F.col("frame_path")) == base(F.col("frame_path_q")))
                .select("frame_sha256", "criticality", "content"))
    t0 = time.time()
    if EXECUTORS_CAN_SCORE:
        scored = to_score.repartition(4).mapInPandas(score_frames, SCORE_SCHEMA)
        where = "on the executors (mapInPandas)"
    else:
        scored = spark.createDataFrame(next(score_frames(iter([to_score.toPandas()]))), SCORE_SCHEMA)
        where = "on the driver (the executors lack numpy / Pillow / scikit-learn)"
    scored = scored.join(frames.select("frame_sha256", "video_id", "asset_id"), "frame_sha256").cache()
    print(f"scored {scored.count()} keyframes {where} in {time.time() - t0:.1f} s")
    display(scored.groupBy("band").count().orderBy("band").toPandas())
    scored.select("video_id", "asset_id", "criticality", "band", F.round("p_severe", 3).alias("p_severe"),
                  "severity", "failed_guardrails").orderBy("video_id", "band").toPandas()

**Read it like an engineer.** `NA` frames failed an input guardrail (blur, glare, low light, or unlike
anything in training) and go to calendar-order review: these are the planted unfit frames, now measured on
the pixels instead of guessed from the JPEG size. `UNCERTAIN` sits near the threshold; criticality-A assets
never drop below P2. The model ranks and suggests; an engineer decides.

The scores go to the scratch catalog, as `gold.fact_keyframe_score` would in production, and one query
turns them into a per-asset worklist.

In [ ]:
if HAVE_CHAMPION:
    KS = f"{DEMO}.drone.keyframe_score"
    (scored.withColumn("business_date", F.lit(DEMO_DATE).cast("date"))
     .withColumn("champion", F.lit(f"{META['estimator']}:{META['git_sha'][:7]}"))
     .writeTo(KS).using("iceberg").createOrReplace())
    worklist = spark.sql(f'''
        SELECT asset_id, criticality, count(*) AS frames,
               sum(CASE WHEN band = 'P1' THEN 1 ELSE 0 END) AS p1,
               sum(CASE WHEN band = 'P2' THEN 1 ELSE 0 END) AS p2,
               sum(CASE WHEN band = 'NA' THEN 1 ELSE 0 END) AS unfit,
               round(max(p_severe), 3) AS worst_p_severe
        FROM {KS} GROUP BY asset_id, criticality
        ORDER BY p1 DESC, p2 DESC, worst_p_severe DESC''').toPandas()
    display(worklist)
    ax = worklist.set_index("asset_id")[["p1", "p2", "unfit"]].plot.barh(stacked=True, figsize=(8, 3.5))
    ax.set_title(f"Drone keyframes of {DEMO_DATE} by band, per asset")
    plt.tight_layout()
    plt.show()

> **Say:** "Spark did the plumbing at scale: find the frames, read them where they lie, fan them out. CAI
> brought what Spark alone does not have: an image decoder, a governed model and its guardrails. Same
> functions as the endpoint the engineers use, so the batch score and the live score agree."

## Likely questions

- **"Did this touch production?"** No. Production tables were read; writes went to the `ogx_demo` Hadoop
  catalog under `notebook_scratch/`, which is not in the metastore and is deleted below. No job ran.
- **"Why is the keyframe extractor standard-library Python, not OpenCV?"** The CDE jobs run without a
  python-env, and the drone writes Motion-JPEG, so a keyframe is a change in JPEG bytes. Real H.264 video
  needs ffmpeg / PyAV in a CDE python-env or a CAI session (docs/DRONE_VIDEO_SPARK.md).
- **"Why mapInPandas and not a UDF per row?"** The model loads once per partition and scores the batch;
  a row UDF would unpickle it for every frame.
- **"Is this in production?"** Not yet: the nightly batch score (`ogx-06`, `gold.fact_keyframe_score`) is
  planned. Today the Workbench scores frames on demand through the endpoint, and `ogx-05` scores them
  nightly for drift.
- **"The CAI endpoint reads frames from the committed library; this read `raw/frames/`?"** Yes. The
  generated videos are built from the library, so both find the same bytes by hash; reading `raw/frames/`
  is what a real deployment does.
- **"Is the data real?"** The photographs are real (CC-BY-4.0, `assets/frames/ATTRIBUTION.md`); the
  operator, assets, flights and faults are synthetic. Say so.

## Clean up

Drops the scratch tables, deletes the scratch folder and stops the session.

In [ ]:
for tbl in (f"{DEMO}.drone.keyframe_score", f"{DEMO}.drone.flight_summary"):
    spark.sql(f"DROP TABLE IF EXISTS {tbl} PURGE")
root = spark._jvm.org.apache.hadoop.fs.Path(SCRATCH)
hfs = root.getFileSystem(spark._jsc.hadoopConfiguration())
if hfs.exists(root):
    hfs.delete(root, True)
print("scratch removed:", not hfs.exists(root))
spark.stop()